In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import bbknn
import scrublet as scr
import matplotlib.pyplot as plt
import scanpy.external as sce
import hotspot

sc.settings.set_figure_params(dpi=1000,figsize=(5, 5))
sc.logging.print_header()

In [ ]:
## data dir
input_dir = " "
output_dir = " "
all_tumor_code = os.listdir(input_dir)

## pre work
regress = True
cluster_method = "leiden"
bbknn_ridge = True
resolution = 2
random_state = 123
n_iterations = -1


In [ ]:
all_tumor_code

In [ ]:
## data process
scRNA_current = []
for tumor_code in all_tumor_code:
    print(f"      ----------------- {tumor_code}  read")

    ## data read
    tumor_current = diopy.input.read_h5(file = f"{input_dir}/{tumor_code}/scRNA_annotation_new.h5")

    ### data add
    scRNA_current.append(tumor_current)
    
    del tumor_current
    gc.collect()

## data merge
scRNA_current = sc.concat(scRNA_current)
scRNA_current.obs_names_make_unique()

## all cell meta save
all_meta_data = scRNA_current.obs
all_meta_data.to_csv(f"{output_dir}/all_cell_combine_meta.csv", index=False)

## origin data save
scRNA_current.write_h5ad(f"{output_dir}/scRNA_all_cell_type_origin.h5ad", compression="gzip")
diopy.output.write_h5(scRNA_current, file = f"{output_dir}/scRNA_all_cell_type_origin.h5",save_X=False)

## tumor cell and normal save
scRNA_Epithelia = scRNA_current[scRNA_current.obs["new_cell_type"].isin(["Epithelia","Epithelia_tumor","Osteoblastic_tumor","Melanoma_tumor","Melanoma_cell","Osteoblastic_cell"]),:]
diopy.output.write_h5(scRNA_Epithelia, file = f"{output_dir}/scRNA_Epithelia.h5",save_X=False)
del scRNA_Epithelia
gc.collect()

## data nor
scRNA_current.layers["counts"] = scRNA_current.X.copy()
sc.pp.normalize_total(scRNA_current, target_sum=1e4)
sc.pp.log1p(scRNA_current)
scRNA_current.layers["log1p"] = scRNA_current.X.copy()
scRNA_current.raw = scRNA_current

## data save
scRNA_current.write_h5ad(f"{output_dir}/scRNA_all_cell_type_cluster.h5ad", compression="gzip")
diopy.output.write_h5(scRNA_current, file = f"{output_dir}/scRNA_all_cell_type_cluster.h5",save_X=False)


In [ ]:
scRNA_current = sc.read_h5ad(f"{output_dir}/scRNA_all_cell_type_cluster.h5ad")

In [ ]:
## cell group transform
scRNA_current.obs["new_cell_group"] = scRNA_current.obs["cell_type"]
scRNA_current.obs.loc[scRNA_current.obs["final_cell_type"].str.startswith("T_"), "new_cell_group"] = "T_cell"
scRNA_current.obs.loc[scRNA_current.obs["final_cell_type"].str.startswith("NK"), "new_cell_group"] = "NK_cell"


In [ ]:
scRNA_current = scRNA_current[scRNA_current.obs.new_cell_group!="Bela_cell", :]

In [ ]:
gc.collect()
## marker plot
marker_gene = {
    "T_cell": ["CD3D","CD3E","CD2"],
    "NK_cell": ["NKG7","GNLY","KLRF1"], 
    "B_cell": ["CD79A","CD79B","MS4A1"],
    "Plasma_cell": ["IGHA1","MZB1","IGHG1"], 
    "Myeloid_cell": ["LYZ","AIF1","C1QC"], 
    "Neutrophils": ["CSF3R","IL1R2","CXCL8"], 
    "Mast_cell": ["TPSAB1","CPA3","TPSB2"], 
    # "Fibroblast": ["DCN","LUM"],
    "Fibroblast": ["DCN","COL3A1","COL6A1"],
    "SMC&Pericyte":["MYH11","RGS5","ACTA2"],
    "Epithelia": ["EPCAM","KRT18","KRT19"], 
    "Endothelia": ["PECAM1","VWF","PLVAP"], 
    "Glial_cell":["S100B","PLP1"],
    "Neuron":["MAP2","STMN2"],
    "Hepatocyte":["ALB","FABP1","CYP2E1"],
    "Melanoma_cell":["MLANA","PMEL","DCT"],
    "Acinar_cell":["CPA1","CTRB1"],
    # "Bela_cell":["MAFA","DLK1","NPTX2"],
    "Osteoblastic_cell":["ALPL","RUNX2","CLEC11A"]
}
sc.pl.dotplot(scRNA_current, marker_gene, 'new_cell_group',categories_order=marker_gene.keys(),show=False,dendrogram=False,vmax = 3, vmin = -3)
plt.savefig(f'{output_dir}/dotplot_marker.svg', dpi=500)

In [ ]:
sc.pl.matrixplot(scRNA_current, marker_gene, 'new_cell_group',categories_order=marker_gene.keys(),show=False,dendrogram=False,cmap='RdBu_r',use_raw=False,vmax = 3, vmin = -3)
plt.savefig(f'{output_dir}/pheatmap_marker.svg', dpi=500)